In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import pandas as pd
import importlib
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

# Infrastructure parameters
POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

In [ ]:
window_days = 7
window = window_days * 24 * 60
test_days = 180
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

In [ ]:
pd.read_sql(select(models.ProviderAssetGroupMember), engine).to_parquet(
    "provider_asset_group_member.parquet"
)

In [ ]:
pd.read_sql(
    select(models.ProviderAssetMarket)
    .where(
        models.ProviderAssetMarket.timestamp.between(start_time, end_time),
    )
    .order_by(models.ProviderAssetMarket.timestamp),
    engine,
).to_parquet("provider_asset_market.parquet")